In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

import qwen_vl

/home/guests/nicolas_stellwag/surgery-scene-graphs/.pixi/envs/default/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
graph_root = Path("output/initial/video17_01803/graph")

In [3]:
model, processor = qwen_vl.get_patched_qwen()

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [4]:
systemprompt = """
  You are an expert visceral surgeon analyzing a cholecystectomy image.
  """

### Static

In [5]:
feats = np.load(graph_root / "c_qwen_feats.npz")
adjacencies = np.load(graph_root / "graph.npy")
centers = np.load(graph_root / "c_centers.npy")
centroids = np.load(graph_root / "c_centroids.npy")
extents = np.load(graph_root / "c_extents.npy")

In [6]:
qwen_vl.prompt_with_descriptors_at_timestep(
    question="Count how many descriptors you receive in this prompt. Does every descriptor have a corresponding image? Go over all descriptors and describe what you see in its image.",
    node_feats=feats,
    timestep_idx=0,
    model=model,
    processor=processor,
    system_prompt="You are a helpful assistant.",
)

'There are 8 descriptors in the provided prompt. Each descriptor has a corresponding image, as follows:\n\n1. **Descriptor ID: 0**\n   - The image shows a close-up of a surgical instrument, likely a needle or a similar tool, held by a hand. The background is blurred, focusing attention on the instrument.\n\n2. **Descriptor ID: 1**\n   - This image appears to be the same as the first one, showing the same surgical instrument with a slightly different angle or focus.\n\n3. **Descriptor ID: 2**\n   - The image shows a surgical procedure inside a body cavity, possibly during an operation. There is a visible surgical instrument, but the focus is more on the surrounding tissue and blood vessels.\n\n4. **Descriptor ID: 3**\n   - Similar to the second image, this one also shows a surgical procedure inside a body cavity, with a surgical instrument and visible blood vessels.\n\n5. **Descriptor ID: 4**\n   - This image is identical to the third one, showing the same surgical procedure with the sa

In [7]:
qwen_vl.prompt_with_graph_at_timestep(
    question="Count how many graph nodes you receive in this prompt. Does every graph node have a corresponding image? Go over all graph nodes and describe what they are.",
    node_feats=feats,
    timestep_idx=0,
    adjacency_matrices=adjacencies,
    node_centers=centers,
    node_centroids=centroids,
    node_extents=extents,
    model=model,
    processor=processor,
    system_prompt="You are a helpful assistant.",
)



"In the provided spatial graph, there are 8 nodes (nodes with IDs from 0 to 7). Each node has a corresponding image, as indicated by the `<descriptor>` tag within each node's XML structure.\n\nHere is a description of each node based on its descriptor:\n\n1. **Node ID: 0**\n   - Descriptor: The image shows a surgical instrument, likely a needle or a similar tool, being used for suturing or another surgical procedure.\n   - Centroid: (-0.04, 0.01, 1.37)\n   - Extent: (0.25, 0.25, 0.27)\n\n2. **Node ID: 1**\n   - Descriptor: The image shows a close-up view of internal organs, possibly during a surgical procedure. A surgical instrument is visible, but it is not as prominent as in Node 0.\n   - Centroid: (-0.11, -0.02, 1.46)\n   - Extent: (0.20, 0.16, 0.19)\n\n3. **Node ID: 2**\n   - Descriptor: The image shows a close-up view of internal organs, similar to Node 1, with a surgical instrument partially visible.\n   - Centroid: (-0.11, -0.19, 1.19)\n   - Extent: (0.38, 0.19, 0.20)\n\n4. **No

In [11]:
len(feats.keys())

8

'The provided scene graph describes a series of spatial relationships between objects over time, specifically focusing on a cholecystectomy procedure. The objects in the scene represent various anatomical structures and surgical instruments involved in the operation.\n\n### Scene Description:\n\n#### Timepoint 0:\n- **Object 0**: Appears to be a surgical instrument, possibly a clamp or forceps, positioned near the gallbladder (Object 1). It is located at coordinates (-0.25, 0.26, 0.91).\n- **Object 1**: Represents the gallbladder, which is being manipulated by Object 0. It is located at coordinates (0.13, 0.15, 0.91).\n- **Object 2**: Likely represents another anatomical structure, such as the liver or surrounding tissue, located at coordinates (0.24, 0.13, 0.76).\n- **Object 3**: Another anatomical structure, possibly the duodenum or another part of the digestive tract, located at coordinates (0.18, 0.10, 0.84).\n- **Object 4**: A surgical instrument, possibly a scalpel or scissors, located at coordinates (0.15, -0.06, 0.91).\n- **Object 5**: Another surgical instrument, possibly a probe or retractor, located at coordinates (-0.21, 0.03, 0.94).\n\n#### Timepoints 1 through 19:\n- The spatial relationships between these objects remain relatively consistent across the timepoints, indicating that the surgical instruments are moving around the gallbladder and other anatomical structures.\n- The centroids and extents of each object provide information about their positions and dimensions. For example, Object 0 (the clamp) moves slightly but remains close to the gallbladder (Object 1), suggesting it is being used to manipulate or hold the gallbladder in place.\n- Object 4 (the scalpel/scissors) appears to be in motion, possibly cutting or dissecting tissues, while Object 5 (the probe/retractor) seems to be stabilizing or retracting parts of the anatomy.\n- The extent values for each object indicate the size and spread of the object in space, which helps in understanding the spatial relationship between different anatomical structures and instruments.\n\n### Key Observations:\n- The gallbladder (Object 1) is the central focus, with the surgical instruments (Objects 0, 4, and 5) interacting with it.\n- The movement of the instruments suggests a step-by-step dissection process, where the gallbladder is being isolated and prepared for removal.\n- The consistent positioning of the anatomical structures indicates a controlled and precise surgical environment.\n\n### Conclusion:\nThroughout the sequence of timepoints, the scene depicts a detailed surgical procedure involving the manipulation of the gallbladder using various surgical instruments. The spatial relationships between the objects suggest a methodical approach to the cholecystectomy, with the instruments carefully positioned to ensure precision and safety during the operation.'

In [ ]:
qwen_vl.prompt_with_dynamic_graph(
    question="Describe the scene throughout time in detail.",
    node_feats=feats,
    adjacency_matrices=adjacencies,
    node_centers=centers,
    node_centroids=centroids,
    node_extents=extents,
    model=model,
    processor=processor,
    system_prompt=systemprompt,
)

"The provided scene graph describes a series of spatial graphs representing different time points (t=0 to t=19) during a cholecystectomy procedure, focusing on the surgical field and instruments involved. Here's a detailed description of the scene throughout the time points:\n\n### Initial Scene (t=0):\n- The initial frame shows a close-up view of the surgical site, likely the gallbladder area, with visible tissue and blood vessels.\n- A surgical instrument, possibly a forceps or clamp, is positioned near the gallbladder, indicating the start of the procedure.\n\n### Progression Through Time:\n1. **Surgical Instrument Movement**: Throughout the frames, the surgical instrument appears to be moving slightly, suggesting manipulation of the gallbladder or surrounding tissues.\n2. **Tissue Changes**: The tissue around the gallbladder seems to be slightly manipulated or retracted, indicating the progression of the dissection process.\n3. **Blood Vessels**: Blood vessels are consistently visible, showing the ongoing nature of the surgery and the need for careful hemostasis.\n4. **Instrument Positioning**: The instrument's position changes subtly across the frames, reflecting the surgeon's precise movements as they work to remove the gallbladder.\n\n### Key Observations:\n- **Steady Progression**: The frames show a steady progression of the surgical procedure, with the instrument gradually moving closer to the gallbladder.\n- **Attention to Detail**: The focus remains on the surgical site, highlighting the precision required for such a delicate operation.\n- **Continuous Manipulation**: The instrument continues to interact with the tissue, suggesting that the surgeon is carefully dissecting and preparing the area for the final removal of the gallbladder.\n\n### Final Scene (t=19):\n- By the end of the sequence, the surgical instrument is still present but has moved further away from the gallbladder, indicating that the dissection phase may have concluded.\n- The tissue around the gallbladder appears more retracted, suggesting that the surgeon is nearing the completion of the procedure.\n\nOverall, the scene graph provides a detailed and continuous view of the cholecystectomy process, emphasizing the meticulous nature of the surgery and the importance of precise instrumentation and tissue manipulation."